# Experiment 2 — Context Ladder for Real-Bug Detection with a Code LLM

**PhD topic:** Large Language Models for Automated Program Debugging · **Builds on:** Experiment 1 (`BugInPy_CodeBERT_Experiment.ipynb`, results in `RESULTS_experiment1.md`) · **Plan:** `EXPERIMENT2_PLAN.md`

Experiment 1 showed that fine-tuned CodeBERT detects real BugInPy bugs at chance level from the function text alone (ROC-AUC ≈ 0.50), while the same pipeline detects injected obvious bugs perfectly. Experiment 2 asks **which context a code LLM needs to detect real bugs**, and **whether its successes come from understanding or from memorising BugInPy**.

| RQ | Question |
|---|---|
| RQ1 | Does a code LLM, zero-shot, detect real bugs from the function alone (C0) better than chance and than CodeBERT? |
| RQ2 | Which context makes real bugs detectable: specification (C1), file skeleton (C2), failing test (C3)? |
| RQ3 | Is success due to memorisation (identifier-renaming probe)? |
| RQ4 | Sanity: does the LLM pipeline detect injected obvious bugs (positive control)? |

## How to run (lessons from Experiment 1)
1. Use a **GPU runtime (T4)**. Check that the bottom bar shows `T4`.
2. In a **new session**, run **E2-1.1 (installation) FIRST**. If Colab asks to *Restart session*, accept, then continue with E2-1.2. Do **not** run E2-1.1 again in the same session.
3. Then run the cells **in order, one by one**.
4. Everything is saved on Drive under `PhD_Experiments/BugInPy_CodeBERT/exp2/`.

| Phase | Content | Status |
|---|---|---|
| E2-1 | Setup, model loading, scoring function, smoke test | **available** |
| E2-2 | Evaluation subset, context ladder C0–C3, leakage controls, renaming | pending |
| E2-3 | Prompt selection on the dev set | pending |
| E2-4 | Positive control | pending |
| E2-5 | Inference (resumable) | pending |
| E2-6 | Evaluation | pending |
| E2-7 | Analysis & report | pending |

---
# PHASE E2-1 — Setup, model and scoring function

**Goal:** a reproducible environment, the code LLM loaded in 4-bit on a T4, and a **deterministic scoring function** that turns the LLM's answer into a probability.

**Scoring principle.** The prompt ends with *"Does this function contain a bug? Answer with a single word: Yes or No."*. Instead of letting the model write an answer, we read the probabilities of the next token:

> score = P(Yes) / (P(Yes) + P(No))

This needs **one forward pass** (fast), involves **no sampling** (deterministic), and gives a probability, so ROC-AUC and paired comparisons can be computed exactly as in Experiment 1. We also record `mass_yes_no` = P(Yes) + P(No). If it is low, the model wanted to answer something other than Yes/No, and the prompt must be improved (Phase E2-3).

## E2-1.1 Installation (run FIRST in every new session)

**Explanation.** Installs `bitsandbytes` (4-bit quantisation, needed to fit a 7-billion-parameter model on a 15 GB T4), and makes sure that `transformers` and `accelerate` are compatible.

If packages were changed, the cell **restarts the Python session automatically**. Colab then shows *"Your session crashed/restarted"*, which is **expected**. Mixing an already imported old `transformers` with newly installed files causes errors such as `No module named 'transformers.models.…'`.

**Expected output.** pip messages, the versions before/after, then either the automatic restart (continue with **E2-1.2**) or `[OK] E2-1.1`.


In [ ]:
# ---- E2-1.1 Installation (restarts the session automatically when needed) ----
import importlib.metadata as _md, sys as _sys
def _v(p):
    try: return _md.version(p)
    except _md.PackageNotFoundError: return None
_before = {p: _v(p) for p in ("transformers", "accelerate", "bitsandbytes")}
%pip install -q -U "transformers>=4.44,<5" "accelerate>=0.33" "bitsandbytes>=0.43" "scikit-learn>=1.3" tqdm
_after = {p: _v(p) for p in ("transformers", "accelerate", "bitsandbytes")}
print("before:", _before); print("after :", _after)
if _before != _after or "transformers" in _sys.modules:
    print("\nPackages changed (or transformers was already imported): RESTARTING the session automatically.")
    print("When the session has restarted, continue with E2-1.2 (do NOT run E2-1.1 again).")
    import os as _os; _os.kill(_os.getpid(), 9)        # standard Colab way to restart the Python process
print("\n[OK] E2-1.1 — packages already up to date, continue with E2-1.2.")


## E2-1.2 Environment, Drive and experiment configuration

**Explanation.** Imports, seed, GPU information, Google Drive, and the paths of **Experiment 1** (whose data are reused) and **Experiment 2** (new outputs in `exp2/`). The cell checks that the Experiment 1 artefacts needed later are on Drive.

`RESUME_RUN_ID`: leave it `None` the first time. After a restart, put the printed run ID here so that everything is saved under the same run.

**Expected output.** Python and GPU information (`Tesla T4`), the Drive paths, `Experiment 1 artefacts: all found`, the run ID, and `[OK] E2-1.2`.

In [ ]:
# ---- E2-1.2 Environment & configuration ----
import os, sys, platform, json, time, random, re, ast, hashlib, datetime, gc, textwrap, warnings, subprocess
from pathlib import Path
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import numpy as np, pandas as pd
import torch, transformers
import importlib.metadata as _md
if transformers.__version__ != _md.version("transformers"):
    raise RuntimeError(f"transformers in memory ({transformers.__version__}) differs from the installed one "
                       f"({_md.version('transformers')}): use Runtime > Restart session, then run E2-1.2 again.")
from tqdm.auto import tqdm
warnings.filterwarnings("ignore", category=FutureWarning)
transformers.logging.set_verbosity_error()

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed()

IN_COLAB = "google.colab" in sys.modules
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python      :", sys.version.split()[0], "| torch", torch.__version__, "| transformers", transformers.__version__)
print("Device      :", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "no GPU")
if DEVICE.type == "cuda":
    print(f"GPU memory  : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

USE_DRIVE = True
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/PhD_Experiments/BugInPy_CodeBERT")
else:
    BASE_DIR = Path.cwd() / "BugInPy_CodeBERT"
EXP1 = {"processed": BASE_DIR / "data" / "processed", "interim": BASE_DIR / "data" / "interim",
        "raw": BASE_DIR / "data" / "raw", "metrics": BASE_DIR / "results" / "metrics",
        "predictions": BASE_DIR / "results" / "predictions"}
E2 = {k: BASE_DIR / "exp2" / k for k in ("data", "cache", "results", "figures", "logs")}
for p in E2.values(): p.mkdir(parents=True, exist_ok=True)
WORK = (Path("/content") if IN_COLAB else Path.cwd()) / "work_exp2"; WORK.mkdir(parents=True, exist_ok=True)

needed = {"phase5 dataset": EXP1["processed"] / "phase5_dataset_v2.jsonl",
          "phase6 splits": EXP1["processed"] / "phase6_splits.csv",
          "phase4 source files": EXP1["interim"] / "phase4_source_files.tar.gz",
          "BugInPy archive": next(iter(sorted(EXP1["raw"].glob("BugsInPy_*.tar.gz"))), EXP1["raw"] / "BugsInPy_MISSING.tar.gz"),
          "CodeBERT OOF (extended)": EXP1["predictions"] / "phase8_extended_oof_predictions.csv"}
missing = [k for k, p in needed.items() if not p.exists()]
print("\nExperiment 1 artefacts:", "all found" if not missing else f"MISSING {missing}")

RESUME_RUN_ID = None
RUN_ID = RESUME_RUN_ID or "exp2_" + datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
cfg_path = E2["logs"] / f"config_{RUN_ID}.json"
CONFIG2 = json.loads(cfg_path.read_text()) if cfg_path.exists() else {}
CONFIG2.update({"run_id": RUN_ID, "seed": SEED, "base_dir": str(BASE_DIR),
                "hardware": {"device": str(DEVICE), "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None},
                "libraries": {"torch": torch.__version__, "transformers": transformers.__version__, "python": sys.version.split()[0]}})
def save_json(obj, path):
    Path(path).write_text(json.dumps(obj, indent=2, default=str)); return Path(path)
save_json(CONFIG2, cfg_path)
print("RUN_ID      :", RUN_ID, "-> keep it for RESUME_RUN_ID after a restart")
print("Exp. 2 dir  :", BASE_DIR / "exp2")

# ---- Verification ----
assert not missing, f"Experiment 1 files missing on Drive: {missing}"
if DEVICE.type != "cuda":
    print("WARNING: no GPU. Switch to a T4 runtime before E2-1.3 (Runtime > Change runtime type).")
print("\n[OK] E2-1.2")

## E2-1.3 Load the code LLM (4-bit)

**Explanation.** Loads **Qwen2.5-Coder-7B-Instruct** (open weights, instruction-tuned for code) in **4-bit NF4** quantisation with `bitsandbytes`, so that it fits in about 5–6 GB of GPU memory. The first time, about 15 GB are downloaded from Hugging Face (≈ 3–6 minutes). The exact model revision (commit hash) is recorded for reproducibility.

`SMOKE_TEST = True` loads a tiny 0.5B model instead. It is used only to check that the code runs, never for results.

**Expected output.** The model name and revision, its memory footprint (≈ 5–6 GB), and `[OK] E2-1.3`.

In [ ]:
# ---- E2-1.3 Load the LLM ----
_missing = [n for n in ("DEVICE", "CONFIG2", "save_json", "cfg_path") if n not in globals()]
if _missing:
    raise RuntimeError("Run E2-1.2 first in this session (the session was restarted, e.g. after changing the runtime type). "
                       f"Missing: {_missing}")
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
SMOKE_TEST = False
MODEL_ID = "Qwen/Qwen2.5-Coder-0.5B-Instruct" if SMOKE_TEST else "Qwen/Qwen2.5-Coder-7B-Instruct"
if not SMOKE_TEST and DEVICE.type != "cuda":
    raise RuntimeError("A T4 GPU is required for the 7B model: Runtime > Change runtime type > T4 GPU, then re-run E2-1.2 and this cell.")
t0 = time.time()
tok = AutoTokenizer.from_pretrained(MODEL_ID)
if DEVICE.type == "cuda" and not SMOKE_TEST:
    qcfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                              bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    llm = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=qcfg, device_map="auto", torch_dtype=torch.float16)
else:
    llm = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32).to(DEVICE)
llm.eval()
MODEL_REVISION = getattr(llm.config, "_commit_hash", None)
mem = llm.get_memory_footprint() / 1024**3
print(f"Model      : {MODEL_ID}")
print(f"Revision   : {MODEL_REVISION}")
print(f"Footprint  : {mem:.2f} GB | load time {time.time() - t0:.0f}s | context window {llm.config.max_position_embeddings} tokens")
CONFIG2["model"] = {"id": MODEL_ID, "revision": MODEL_REVISION, "quantization": "4-bit NF4 (bitsandbytes)" if DEVICE.type == "cuda" and not SMOKE_TEST else "none",
                    "smoke_test": SMOKE_TEST}
save_json(CONFIG2, cfg_path)

# ---- Verification ----
assert llm.config.max_position_embeddings >= 8192
print("\n[OK] E2-1.3" + ("  (SMOKE TEST model — not for results)" if SMOKE_TEST else ""))

## E2-1.4 Scoring function

**Explanation.** `bug_score(code, context=None)` builds a chat prompt with the model's own template:
* **system:** *"You are an expert Python code reviewer."*
* **user:** optional context sections (filled in Phase E2-2), then the function in a ```python block, then the question.

It then reads the next-token distribution and returns:
* `score` = P(Yes) / (P(Yes) + P(No)), the **bug score**;
* `mass_yes_no` = P(Yes) + P(No), which measures how much the model "wants" to answer Yes/No;
* `n_tokens`, the prompt length.

The Yes/No probabilities sum over the usual spellings ("Yes", "yes", " Yes", …).

**Expected output.** The token ids used for Yes and No, and `[OK] E2-1.4`.

In [ ]:
# ---- E2-1.4 Scoring function ----
_missing = [n for n in ("DEVICE", "CONFIG2", "llm", "tok") if n not in globals()]
if _missing:
    raise RuntimeError("Run E2-1.2 and E2-1.3 first in this session (the session was restarted, e.g. after changing the runtime type). "
                       f"Missing: {_missing}")
SYSTEM_PROMPT = "You are an expert Python code reviewer."
QUESTION = "Does this function contain a bug? Answer with a single word: Yes or No."

def first_ids(words):
    return sorted({tok.encode(w, add_special_tokens=False)[0] for w in words})
YES_IDS = first_ids(["Yes", "yes", " Yes", " yes", "YES"])
NO_IDS  = first_ids(["No", "no", " No", " no", "NO"])
assert not set(YES_IDS) & set(NO_IDS)

def build_user_text(code, context=None, question=QUESTION):
    parts = []
    for title, body in (context or []):
        parts.append(f"### {title}\n{body}")
    parts.append(f"### Function to review\n```python\n{code}\n```")
    parts.append(question)
    return "\n\n".join(parts)

def build_prompt(user_text, system=SYSTEM_PROMPT):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user_text}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def bug_score(code, context=None, question=QUESTION, system=SYSTEM_PROMPT):
    enc = tok(build_prompt(build_user_text(code, context, question), system), return_tensors="pt").to(llm.device)
    logits = llm(**enc).logits[0, -1].float()
    probs = torch.softmax(logits, dim=-1)
    p_yes, p_no = probs[YES_IDS].sum().item(), probs[NO_IDS].sum().item()
    return {"score": p_yes / (p_yes + p_no) if p_yes + p_no > 0 else 0.5,
            "p_yes": p_yes, "p_no": p_no, "mass_yes_no": p_yes + p_no, "n_tokens": int(enc["input_ids"].shape[1])}

print("Yes token ids:", YES_IDS, [tok.decode([i]) for i in YES_IDS])
print("No  token ids:", NO_IDS, [tok.decode([i]) for i in NO_IDS])
print("\n[OK] E2-1.4")

## E2-1.5 Smoke test

**Explanation.** Four tiny hand-written functions, where the answer is obvious to a human:

| # | Function | Expected |
|---|---|---|
| 1 | correct `add(a, b)` | low score |
| 2 | `add` with an injected `1 / 0` | high score |
| 3 | `sum_list` that skips the last element (`range(len(xs) - 1)`) | higher than #4 |
| 4 | correct `sum_list` | low score |

We also check that the score is **deterministic** (same input → same score), that the model answers Yes/No (`mass_yes_no` high), and how long one prompt takes.

**Expected output.** A table with the four scores; for the 7B model, #2 and #3 should get clearly higher scores than #1 and #4. Then the time per prompt (≈ 0.2–1 s on a T4 for short prompts) and `[OK] E2-1.5`.

*This is not an evaluation*, only a check that the scoring behaves sensibly.

In [ ]:
# ---- E2-1.5 Smoke test ----
_missing = [n for n in ("DEVICE", "CONFIG2", "llm", "tok", "bug_score") if n not in globals()]
if _missing:
    raise RuntimeError("Run E2-1.2, E2-1.3 and E2-1.4 first in this session (the session was restarted, e.g. after changing the runtime type). "
                       f"Missing: {_missing}")
examples = {
    "1 correct add":        "def add(a, b):\n    return a + b",
    "2 add + injected 1/0": "def add(a, b):\n    _ = 1 / 0\n    return a + b",
    "3 sum_list off-by-one":"def sum_list(xs):\n    total = 0\n    for i in range(len(xs) - 1):\n        total += xs[i]\n    return total",
    "4 correct sum_list":   "def sum_list(xs):\n    total = 0\n    for i in range(len(xs)):\n        total += xs[i]\n    return total",
}
rows, t0 = [], time.time()
for name, code in examples.items():
    r = bug_score(code); rows.append({"example": name, **r})
dt = (time.time() - t0) / len(examples)
smoke = pd.DataFrame(rows).set_index("example")
display(smoke[["score", "p_yes", "p_no", "mass_yes_no", "n_tokens"]].round(4))
again = bug_score(examples["3 sum_list off-by-one"])["score"]
print(f"Time per prompt: {dt:.2f}s | deterministic: {abs(again - smoke.loc['3 sum_list off-by-one', 'score']) < 1e-4}")
ordering_ok = smoke.loc["2 add + injected 1/0", "score"] > smoke.loc["1 correct add", "score"] and \
              smoke.loc["3 sum_list off-by-one", "score"] > smoke.loc["4 correct sum_list", "score"]
print("Buggy examples scored higher than their correct versions:", ordering_ok)

CONFIG2["scoring"] = {"system": SYSTEM_PROMPT, "question": QUESTION, "yes_ids": YES_IDS, "no_ids": NO_IDS,
                      "smoke_test": smoke.round(4).reset_index().to_dict("records"), "seconds_per_short_prompt": round(dt, 3)}
save_json(CONFIG2, cfg_path)

# ---- Verification ----
assert abs(again - smoke.loc["3 sum_list off-by-one", "score"]) < 1e-4, "scoring is not deterministic"
assert smoke.mass_yes_no.min() > 0.5 or SMOKE_TEST, "the model does not answer Yes/No — prompt problem"
print("\n[OK] PHASE E2-1 COMPLETE" + (" (SMOKE TEST)" if SMOKE_TEST else ""))

**Please report back** the outputs of **E2-1.2** (GPU, artefacts, RUN_ID), **E2-1.3** (model, revision, footprint) and **E2-1.5** (the score table, time per prompt, ordering).

➡️ **PHASE E2-2 (evaluation subset and context ladder)** will be added after these results are confirmed.